# Fase 0: Installazione di librerie

In questa fase sarà possibile vedere le varie librerie installate per questa parte del progetto

In [2]:
%pip install pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# Fase 1: Caricamento e pulizia dei dati

Come prima cosa, ho letto i file di input per i nomi e i cognomi, con lo scopo di creare delle email casuali. Tutti e tre i file sono stati scaricati sotto forma di txt al seguente link: https://github.com/Max1234-Ita/Liste. Ho preferito convertirli in formato csv per puro scopo estetico.

In [3]:
import pandas as pd
dfFemale = pd.read_csv(
    "nomi_f.csv", comment="#", header=None, names=["Nome", "Rarità"]
)
dfMale = pd.read_csv(
    "nomi_m.csv", comment="#", header=None, names=["Nome", "Rarità"]
)
df_cognomi = pd.read_csv("cognomi.csv", comment="#", header=None, names=["Cognome"])
df_nomi = pd.concat([dfFemale, dfMale], ignore_index=True)
df_nomi.sample(10)

,Nome,Rarità
4579,ANNUCCIO,4
7150,MILVANO,3
5355,DIDIMO,3
1515,FILOTEA,4
6344,GINEVRO,4
6132,GABRIEL,2
1111,EMILIETTA,2
2456,LUTGARDA,4
8786,UGOBERTO,2
5535,ELMERICO,4


Fatto questo, ho adesso a disposizione due diversi dataframe:
- df_nomi: contiene una lista di nomi maschili e femminili insieme ad un numero che indica la sua rarità (1 diffuso, 4 raro)
- df_cognmi: contiene una lista con solo cognomi

# Fase 2: Generazione pool di utenti (nome ed email)

Partendo dai due dataframe contenenti vari nomi, posso adesso creare una lista di persone con nomi inventati e, di conseguenza, delle email fatte apposta per loro.
Il primo problema da affrontare è trovare un meccanismo per rendere i nomi più rari meno presenti all'interno della pool. Per fare questo, ho assegnato un peso alle varie categorie. Stessa sorte capita anche ad una pool di domini possibili in base a quanto siano utilizzate.

In [4]:
categorie_nomi = [1, 2, 3, 4]
pesi_categorie_nomi = [0.85, 0.11, 0.03, 0.01]
domini = ["gmail.com", "outlook.it", "hotmail.com", "libero.it", "yahoo.com"]
pesi_domini = [0.55, 0.15, 0.13, 0.13, 0.04]

Altra cosa spesso frequente nelle email è anche la data di nascita dell'utente, quindi utilizzando il metodo random.randint() creiamo un metodo per sorteggiare l'età di ogni singolo utente inventato.

In [5]:
import random
def genera_anno_realistico():
    anno = random.randint(1970, 2007)
    return str(anno)[2:]

Arriva finalmente la parte dove tutti i dati analizzati fino ad ora entrano in gioco. In questo ciclo, vengono estratti casualmente nomi, cognomi, domini e miscelati in maniera casuale con una struttura definita da vari formati rendendo la creazine delle email meno statica ma più realistica.
Da notare inoltre che, nella scelta del nome, non vi è una totale casualità in quanto i nomi meno rari saranno più presenti considerando il controllo del peso (*weights*).

A valle, troviamo la popolazione di una lista *pool_utenti* che conterrà *numero_utenti_unici* nomi inventati e *numero_utenti_unici* email corrispettive.

In [6]:
numero_utenti_unici = 125
pool_utenti = []
for _ in range(numero_utenti_unici):
    cat_nome = random.choices(categorie_nomi, weights=pesi_categorie_nomi)[0]
    df_nomi_filtrato = df_nomi[df_nomi["Rarità"] == cat_nome]
    if df_nomi_filtrato.empty:
        df_nomi_filtrato = df_nomi

    # Conserviamo una versione "pulita" con le iniziali maiuscole
    nome_reale = df_nomi_filtrato["Nome"].sample(n=1).values[0].title().strip()
    cognome_reale = df_cognomi["Cognome"].sample(n=1).values[0].title().strip()
    nome_completo = f"{nome_reale} {cognome_reale}"

    # Versione minuscola e senza spazi per l'indirizzo email
    nome_email = nome_reale.lower().replace(" ", "").replace("'", "")
    cognome_email = cognome_reale.lower().replace(" ", "").replace("'", "")

    dominio = random.choices(domini, weights=pesi_domini)[0]
    anno_nascita = genera_anno_realistico()

    formati = [
        f"{nome_email}.{cognome_email}",
        f"{nome_email}.{cognome_email}{anno_nascita}",
        f"{nome_email[0]}.{cognome_email}",
        f"{nome_email[0]}.{cognome_email}{anno_nascita}",
        f"{cognome_email}.{nome_email}",
        f"{nome_email}_{cognome_email}",
        f"{nome_email}{cognome_email}{anno_nascita}",
    ]
    pesi_formati = [0.20, 0.40, 0.10, 0.10, 0.10, 0.05, 0.05]
    struttura_email = random.choices(formati, weights=pesi_formati)[0]
    email_finale = f"{struttura_email}@{dominio}"

    pool_utenti.append({"user_name": nome_completo, "user_email": email_finale})

# Fase 3: Generazione ticket brevi con lessico tipico casuale

Arrivato a questo punto, sarà necessario creare il corpo principale e più importante della generazione del dataset: la generazione vera e propria dei ticket.
Per farlo, ho creato un json contenente una lista lessicale (che può facilmente essere estesa o adattata a piacimento).

In particolare, il json contiene una divisione per tipoligia di ticket. Come da consegna le divisioni sono:
- Amministrazione
- Tecnico
- Commerciale

Ogni divisione contiene 3 liste, ovvero:
- Verbi
- Sostantivi
- Dettagli del ticket

Procedo dunque nel leggere il json e nel creare un metodo che, prendendo in input il nome del file, la pool di utenti e il numero di ticket da creare, estragga senza rimozione un utente qualsiasi per generare *num_ticket* tickets.

In maniera del tutto casuale, lo script estrae una categoria, un verbo, dei sostantivi e dei dettagli da unire in un body che formerà una frase. 

E' anche presente una logica di assegnazione delle priorità, dove molto banalmente si assegna una priorità più alta o bassa in base alla presenza o meno di determinate parole chiave.

A valle, il metodo unisce tutto questo insieme di dati estratti casualmente in un unico record dove troviamo:
- id: un id incrementale che risulta essere univoco nel ticket in particolare
- user_name: nome e cognome di un utente estratto casualmente dalla pool creata nella fase 2
- user_email: email dello stesso utente estratto casualmente dalla pool creata nella fase 2
- title: titolo del ticket, formato dall'unione del verbo principale + sostantivo usato
- body: il corpo effettivo del ticket
- category: categoria del ticket
- priority: la priorità del ticket

In [7]:
import json
import random
import pandas as pd


def genera_ticket_sintetici_con_utenti(nome_file, pool_utenti, num_ticket=200):
    # 1. Caricamento del lessico
    with open(nome_file, "r", encoding="utf-8") as f:
        lessico = json.load(f)

    categorie = list(lessico.keys())

    # 2. Definizione delle strutture statiche FUORI dal ciclo per risparmiare CPU/Memoria
    keywords_alta = {
        "bloccante",
        "offline",
        "crash",
        "urgente",
        "aiuto",
        "leak",
        "picco di traffico",
        "scaduto",
        "sospendere",
        "penale",
    }
    keywords_media = {
        "errore",
        "errato",
        "sollecitare",
        "sollecito",
        "lento",
        "bug",
        "mancante",
        "disdire",
        "rettificare",
        "stornare",
        "problema",
    }

    pesi_oggetti = [0.20, 0.25, 0.10, 0.15, 0.05, 0.10, 0.10, 0.05]
    pesi_formati = [
        0.15,
        0.20,
        0.15,
        0.10,
        0.05,
        0.10,
        0.10,
        0.11,
        0.01,
        0.01,
        0.01,
        0.01,
        0.01,
    ]

    # Pre-estrazione di massa per gli utenti (molto più veloce di random.choice singolo ripetuto)
    utenti_estratti = random.choices(pool_utenti, k=num_ticket)

    dataset = []

    for i in range(num_ticket):
        utente = utenti_estratti[i]
        cat = random.choice(categorie)

        # Estrazione rapida delle componenti testuali
        v = random.choice(lessico[cat]["verbi"])
        s = random.choice(lessico[cat]["sostantivi"])
        d = random.choice(lessico[cat]["dettagli"])

        # Template per gli oggetti (definiti dinamicamente con le variabili già estratte)
        oggetti = [
            f"{v.capitalize()} {s}",
            f"URGENTE: {v.capitalize()} {s}",
            f"Richiesta supporto - categoria: {cat}",
            f"Segnalazione {s}",
            f"AIUTO - problema con {s}",
            f"Ticket {v.capitalize()} {s}",
            f"Richiesta info {s} {d}",
            f"{s} - {d}",
        ]
        title = random.choices(oggetti, weights=pesi_oggetti)[0]

        # Template per i corpi del testo
        formati = [
            f"Buongiorno,\ncon la presente si richiede cortesemente di {v} {s} {d}.\nRestiamo in attesa di un vostro riscontro urgente.\n\nCordiali saluti.",
            f"Ciao, per favore dovremmo {v} {s} {d} il prima possibile.\nFammi sapere si ci sono problemi. Grazie!",
            f"Salve, si richiede di {v} {s} {d}. Restiamo in attesa di un riscontro urgente.",
            f"Potete {v} {s} {d}?\nGrazie, saluti.",
            f"[NOTIFICA AUTOMATICA]: Rilevata necessità di {v} {s} {d}. Si prega di non rispondere a questa email.",
            f"Gentili colleghi,\nvi ricontatto per sapere se ci sono novità in merito alla richiesta di {v} {s} {d}.\nRimango a disposizione.\n\nBuon lavoro.",
            f"Purtroppo si è reso necessario {v} {s} {d}.\nProcedere appena possibile, blocca il lavoro del team.\nGrazie.",
            f"ciao, bisogna {v} {s} {d} entro oggi se riusciamo... fatemi sapere, grazie",
            f"Per curiosità stavo spulciando {s}, ho scoperto che {v} {s} {d}, e questo è un problema!",
            "ho un problema, quando posso chiamarvi?",
            f"Salve, sono {utente['user_name']}. Ho scoperto che {v} {s}:{d}",
            f"{v} {s} {d}.",
            f"sono {utente['user_name']}, ho un problema critico, ne possiamo parlare il prima possibile?",
        ]
        body = random.choices(formati, weights=pesi_formati)[0]

        # =========================================================================
        # LOGICA DI PRIORITÀ CORRETTA ED EFFICIENTE
        # =========================================================================
        # Uniamo titolo e corpo convertiti in minuscolo per una ricerca realistica delle keyword
        testo_completo = f"{title} {body}".lower()

        # Usiamo i set per l'intersezione di parole (operazione in tempo O(1) estremamente veloce)
        # Controlla se la keyword è presente come sottostringa nel testo
        conteggio_alta = sum(1 for kw in keywords_alta if kw in testo_completo)
        conteggio_media = sum(1 for kw in keywords_media if kw in testo_completo)

        if conteggio_alta >= 2 and conteggio_media <= 1:
            priority = "Alta"
        elif conteggio_media >= 2 and conteggio_alta <= 1:
            priority = "Media"
        elif conteggio_alta > conteggio_media:
            priority = "Alta"
        elif conteggio_media > 0:
            priority = "Media"
        else:
            priority = "Bassa"

        dataset.append(
            {
                "id": f"TKT-{i+1:04d}",
                "user_name": utente["user_name"],
                "user_email": utente["user_email"],
                "title": title,
                "body": body,
                "category": cat,
                "priority": priority,
            }
        )

    # Scrittura finale ottimizzata
    df = pd.DataFrame(dataset)
    df.to_csv("ticket_utenti_simulati.csv", index=False, encoding="utf-8-sig")

    print(
        f"Successo! Creato il file 'ticket_utenti_simulati.csv' con {num_ticket} righe totali."
    )

# Fase 4: Creazione del file contenenti i ticket e visualizzazione

Infine, mando in esecuzione il metodo creato nella fase 3 per creare il file contenente utenti e ticket generati in maniera pseudocasuale.

In [11]:
genera_ticket_sintetici_con_utenti("lessico.json", pool_utenti, 500)

Successo! Creato il file 'ticket_utenti_simulati.csv' con 500 righe totali.


Stampo parte del contenuto del file per vederne il risultato finale:

In [12]:
df_anteprima = pd.read_csv("ticket_utenti_simulati.csv")
df_anteprima.head(5)["body"]

0    Ciao, per favore dovremmo riscontrare un bug b...
1    [NOTIFICA AUTOMATICA]: Rilevata necessità di s...
2    Salve, si richiede di disdire il listino prezz...
3    Potete sollecitare lo storno per il servizio c...
4    Purtroppo si è reso necessario contabilizzare ...
Name: body, dtype: str

In [10]:
import pandas as pd

df_anteprima = pd.read_csv("ticket_utenti_simulati.csv")

# Questo ti mostra il numero esatto di ticket per ogni priorità
print(df_anteprima["priority"].value_counts())

priority
Alta     229
Media    167
Bassa    104
Name: count, dtype: int64


## Fase 5: Ulteriore ottimizzazione del dataset

Dalla fase 4 è evidente che già la creazione randomica del dataset da un buon risultato, ma si può fare di meglio. Il rischio attuale è che utilizzare un dataset con quella struttura creata in maniera molto rigida con poche varianti non permetta al modello LLM di imparare a fare un triage basato sul significato del problema del ticket, bensì impari a riconoscere le categorie solamente in base alle frasi fisse; ad esempio, se è presente la keyword "bug bloccante", allora il modello potrebbe capire che possa essere solamente un ticket tecnico, cosa che non è così nella vita reale! Nei prossimi step addestrerò il modello anche con il dataset così composto per verificare se il mio dubbio sia lecito, ma intanto ho deciso di introdurre dell'ambiguità nel testo, creando delle frasi che introduca una percentuale di ticket con un vocabolario misto e a simulare l'errore umano inserendo rumore (ovvero typo, refusi nel testo, diverso registro linguistico).

A proposito di questo, alla fine creerò 3 diversi modelli con diversi dataset:
- Dataset grezzo creato con lo script della fase 3
- Dataset ottimizzato creato con lo script a seguire descritto nella fase 5
- Dataset generato da un modello di Intelligenza Artificiale generativa (Gemini)